# Evex Initial Data Analysis

In [ ]:
import os
import json
import locale
import pickle
from datetime import UTC, datetime, time, timedelta

import numpy as np
import pandas as pd

# from utils import fetch_all_issues, run_prompt, get_summary
import plotly.express as px
import plotly.graph_objects as go
import pytz
from dotenv import load_dotenv
from jira import JIRA
from plotly.subplots import make_subplots

%load_ext autoreload
%autoreload 2
locale.setlocale(locale.LC_TIME, "de_DE.UTF-8")
load_dotenv()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


True

In [ ]:
from data_transformation import load_issues, load_issues_Amparex
from jira_loader import fetch_jira_issues, jira_request

In [ ]:
start_date = datetime.now(UTC) - timedelta(days=600)
end_date = datetime.now(UTC)

tz = pytz.UTC

start_dt = tz.localize(datetime.combine(start_date, time.min))
end_dt = tz.localize(datetime.combine(end_date, time.max))

In [57]:
amparex_cloud_id = {"cloudId": "242cf880-c51a-4277-9381-781d5ae181df"}
amparex_sandbox_cloud_id = {"cloudId": "8a3828c5-f874-43ce-9367-3d9b73c02832"}
issues_ipro = fetch_jira_issues(start_dt, end_dt, max_issues=10000, project="SDIPR")
json.dump(issues_ipro, open("data/issues_ipro.json", "w"), indent=2, ensure_ascii=False)

Total issues fetched: 5741


In [88]:
issues_ax = fetch_jira_issues(start_dt, end_dt, max_issues=50000, project="SDAX")

Total issues fetched: 16393


In [ ]:
print(json.dumps(issues_ipro[:2], indent=2, ensure_ascii=False))

In [ ]:
jira_request("https://amparex.atlassian.net/rest/servicedeskapi/assets/workspace")

In [ ]:
jira_request(
    "https://amparex-sandbox-600.atlassian.net/rest/servicedeskapi/assets/workspace"
)

In [ ]:
from jira_loader import fetch_all_object_schemas

schemas = fetch_all_object_schemas(workspace_id="8a799a44-1189-445f-9b88-56372496d3f0")
schemas

In [ ]:
issues_ax = fetch_jira_issues(start_dt, end_dt, max_issues=50000, project="SDAX")
json.dump(issues_ax, open("data/issues_ax.json", "w"), indent=2, ensure_ascii=False)

In [ ]:
df_ax = load_issues_Amparex(issues_ax)

In [ ]:
df_ax.to_csv("jira_tickets_ax.csv", sep=";", index=False, encoding="utf-8")

In [ ]:
def get_country_counts_service_desk(issues: dict):
    country_ticket_counts = {"No Country": 0}
    for issue in issues:
        if issue.get("customer_country") in country_ticket_counts.keys():
            country_ticket_counts[issue.get("customer_country")] += 1
        elif not issue.get("customer_country"):
            country_ticket_counts["No Country"] += 1
        else:
            country_ticket_counts[issue.get("customer_country")] = 1
    return country_ticket_counts

In [ ]:
country_ticket_count_ipro = get_country_counts_service_desk(issues=issues_ipro)
country_ticket_count_ipro

In [ ]:
country_ticket_count_ax = get_country_counts_service_desk(issues=issues_ax)
country_ticket_count_ax

In [ ]:
# get jira Ticket country by month for ipro and ax, create a df with columns month, year, country, count, project
def get_monthly_country_counts(issues: dict, country_columns: list | None = None):
    if not country_columns:
        country_columns = ["No Country"]
    # Build a matrix: one row per month_year, one column per country, values = ticket count.
    monthly_country_counts = []
    for issue in issues:
        created_date = datetime.strptime(
            issue.get("fields", {}).get("created", ""), "%Y-%m-%dT%H:%M:%S.%f%z"
        )
        month_year = created_date.strftime("%Y-%m")
        country = (
            issue.get("customer_country")
            if issue.get("customer_country")
            else "No Country"
        )
        monthly_country_counts.append({"month_year": month_year, "country": country})
    df_long = pd.DataFrame(monthly_country_counts)
    # Pivot the long counts into a wide matrix indexed by month_year with one column per country.
    df_matrix = df_long.groupby(["month_year", "country"]).size().unstack(fill_value=0)
    # Guarantee every requested country is present as a column, even months with zero tickets.
    for col in country_columns:
        if col not in df_matrix.columns:
            df_matrix[col] = 0
    df_matrix = df_matrix.sort_index().reset_index()
    df_matrix.columns.name = None
    return df_matrix

In [ ]:
df_monthly_country_counts_ipro = get_monthly_country_counts(
    issues=issues_ipro, country_columns=list(country_ticket_count_ipro.keys())
)
df_monthly_country_counts_ipro.to_csv(
    "data/monthly_country_counts_ipro.csv", index=False
)

In [ ]:
df_monthly_country_counts_ax = get_monthly_country_counts(
    issues=issues_ax, country_columns=list(country_ticket_count_ax.keys())
)
df_monthly_country_counts_ax.to_csv("data/monthly_country_counts_ax.csv", index=False)

In [ ]:
len(issues_ipro), len(issues_ax)

In [ ]:
df_ax.columns

In [ ]:
df_ax.issuetype.value_counts()

In [ ]:
df_ax["comments"] = ""

for i in range(len(df_ax)):
    try:
        df_ax.loc[i, "comments"] = str(issues_ax[i]["fields"]["comment"])
    except Exception:  # noqa: BLE001 - deliberate catch-all in exploratory code
        print(i)
        df_ax.loc[i, "comments"] = ""

In [ ]:
df_ax[df_ax["comments"] != ""]

In [ ]:
issues_ipro[0]

In [ ]:
df_new_ipro = load_issues(issues_ipro)

In [ ]:
df_new_ipro["reporter"] = ""
for i in range(len(df_new_ipro)):
    try:
        df_new_ipro.loc[i, "reporter"] = issues_ipro[i]["fields"]["reporter"][
            "emailAddress"
        ]
    except Exception:  # noqa: BLE001 - deliberate catch-all in exploratory code
        df_new_ipro.loc[i, "reporter"] = ""

In [ ]:
df_new_ipro.head()

In [ ]:
df_new_ipro.source.value_counts()

In [ ]:
# load json file
with open("jira_user_org_mapping.json", "r") as f:
    jira_user_mapping = json.load(f)

# turn jira_user_mapping into dataframe
df_jira_user_mapping = pd.DataFrame(jira_user_mapping)
df_jira_user_mapping.head()

In [ ]:
df_new_ipro = df_new_ipro.merge(
    df_jira_user_mapping, left_on="reporter", right_on="emailAddress", how="left"
)

In [ ]:
df_new_ipro["Kunde"] = df_new_ipro["organizationName"].str.extract(r"^(.*?)-\d+-")

In [ ]:
df_new_ipro.head()

In [ ]:
df_new_ipro.to_csv("jira_tickets_ipro.csv", sep=";", index=False, encoding="utf-8-sig")

In [ ]:
df_new_ipro.Kunde.value_counts()

In [ ]:
df_new_ipro[["zentrale", "filiale"]]

In [ ]:
df_new_ipro.zentrale.value_counts()

In [ ]:
df_new_ipro[df_new_ipro["zentrale"] == "ID_9785"][["zentrale", "filiale"]]

## Pull Issue list

In [ ]:
# Create a Jira client and authenticate with API key
# basic_auth = ("bl@flex.capital", "...")
basic_auth = "..."

# jira = JIRA(server="https://amparex.atlassian.net/",basic_auth=basic_auth)
jira = JIRA(server=os.getenv("JIRA_URL"), token_auth=os.getenv("JIRA_API_KEY"))

In [ ]:
jira.projects()

In [ ]:
jql = "project = EXIPR ORDER BY created DESC"
page = jira.enhanced_search_issues(
    jql_str=jql,
    maxResults=20,  # per API call
    json_result=True,
)
page

In [ ]:
jql = "project = EXIPR ORDER BY created DESC"
# issues = fetch_all_issues(jira, jql, max_issues=5000)
issues = fetch_jira_issues(
    max_issues=5000, project="EXIPR", save_path="data/issues_exipr.json"
)

In [ ]:
run_jira_pull = True
if run_jira_pull:
    jql = "project = EXIPR ORDER BY created DESC"
    issues = fetch_jira_issues(max_issues=5000, project="EXIPR", save_path=None)
    # issues = fetch_all_issues(jira, jql, max_issues=5000)
    # dump issues into pickle file
    with open("issues.pkl", "wb") as f:
        pickle.dump(issues, f)

else:
    # retrieve issues from pickle file
    with open("issues.pkl", "rb") as f:
        issues = pickle.load(f)

In [ ]:
len(issues)

In [ ]:
issues[0]["fields"]

In [ ]:
# read json file "fields.json"
with open("response_ticket.json", "r") as f:
    fields = json.load(f)

# save back with indent 4
with open("response_ticket.json", "w") as f:
    json.dump(fields, f, indent=4)

In [8]:
# read json from data/jira-servicedesk-schema-objects.json
with open("data/jira-servicedesk-schema-objects.json", "r") as f:
    schema = json.load(f)

object_id_to_name = {v["id"]: v["name"] for v in schema["values"]}

In [ ]:
df = {
    "key": [],
    "summary": [],
    "description": [],
    "status": [],
    "status_category": [],
    "created": [],
    "updated": [],
    "labels": [],
    "source": [],
    "priority": [],
    "category": [],
    "issuetype": [],
    "main_category_id": [],
    "sub_category_id": [],
    "currentstatus_name": [],
    "currentstatus_date": [],
    "comments": [],
    "request_type": [],
}
for issue in issues:
    df["key"].append(issue["key"])
    df["summary"].append(issue["fields"]["summary"])
    df["description"].append(issue["fields"]["description"])
    df["status"].append(issue["fields"]["status"]["name"])
    df["status_category"].append(issue["fields"]["status"]["statusCategory"]["name"])
    # df['creator'].append(issue['fields']['creator']['displayName'])
    df["issuetype"].append(issue["fields"]["issuetype"]["name"])
    df["created"].append(issue["fields"]["created"])
    df["updated"].append(issue["fields"]["updated"])
    df["labels"].append(issue["fields"]["labels"])
    df["priority"].append(issue["fields"]["priority"]["name"])
    df["category"].append(issue["fields"]["customfield_10065"])

    if issue["fields"]["customfield_10010"] is not None:
        df["request_type"].append(
            issue["fields"]["customfield_10010"]["requestType"]["name"]
        )
    else:
        df["request_type"].append("")
    if issue["fields"]["comment"] is not None:
        df["comments"].append(
            "\n\n".join([c["body"] for c in issue["fields"]["comment"]["comments"]])
        )
    else:
        df["comments"].append([])

    try:
        df["currentstatus_name"].append(
            issue["fields"]["customfield_10010"]["currentStatus"]["status"]
        )
        df["currentstatus_date"].append(
            issue["fields"]["customfield_10010"]["currentStatus"]["statusDate"]["jira"]
        )
    except Exception:  # noqa: BLE001 - deliberate catch-all in exploratory code
        df["currentstatus_name"].append("")
        df["currentstatus_date"].append("")

    try:
        v = issue["fields"]["customfield_10675"]["value"]
        df["source"].append(v)
    except Exception:  # noqa: BLE001 - deliberate catch-all in exploratory code
        df["source"].append("")

    cf = issue["fields"]["customfield_10680"]
    if len(cf) > 0:
        df["main_category_id"].append(cf[0]["objectId"])
    else:
        df["main_category_id"].append("")
    cf = issue["fields"]["customfield_10679"]
    if len(cf) > 0:
        df["sub_category_id"].append(cf[0]["objectId"])
    else:
        df["sub_category_id"].append("")


df = pd.DataFrame(df)
### convert created, updated to datetime
df["created"] = pd.to_datetime(df["created"], errors="coerce", utc=True)
df["updated"] = pd.to_datetime(df["updated"], errors="coerce", utc=True)
df["currentstatus_date"] = pd.to_datetime(
    df["currentstatus_date"], errors="coerce", utc=True
)
df["time_to_resolution_h"] = (
    df["currentstatus_date"] - df["created"]
).dt.total_seconds() / 3600
df["time_to_resolution_days"] = (df["currentstatus_date"] - df["created"]).dt.days
df["resolution"] = "> 1 day"
df["resolution"] = np.where(df["time_to_resolution_days"] <= 1, "Same day", "> 1 day")
df["bdays"] = np.busday_count(
    df["created"].to_numpy(dtype="datetime64[D]"),
    df["updated"].to_numpy(dtype="datetime64[D]"),
)
df["created_string"] = df["created"].dt.strftime("%Y-%m-%d")
df["updated_string"] = df["updated"].dt.strftime("%Y-%m-%d")
df["year"] = df["created"].dt.year
df["month"] = df["created"].dt.month
df["Hauptkategorie"] = df["main_category_id"].map(object_id_to_name)
df["Unterkategorie"] = df["sub_category_id"].map(object_id_to_name)
# put time to resolution into bins
bins = [0, 1, 2, 4, 8, 24, 48, 72, 7 * 24, 14 * 24, 21 * 24]
df["time_to_resolution_bin"] = pd.cut(df["time_to_resolution_h"], bins=bins)
df["time_to_resolution_bin"] = df["time_to_resolution_bin"].apply(
    lambda x: f"{int(x.left)}–{int(x.right)}"
)
df.head(5)

In [ ]:
df.shape

In [ ]:
result = (
    df[df["month"] == 11][["created_string", "key"]]
    .groupby("created_string")
    .count()
    .reset_index()
)
print(result["key"].mean())
# plot using plotly
fig = px.bar(result, x="created_string", y="key", text="key")
# set width of plot
fig.update_layout(width=1000)
# add x label
fig.update_xaxes(title_text="Date")
# add y label
fig.update_yaxes(title_text="Count of tickets")
fig.show()

In [ ]:
df["request_type"].value_counts()

In [ ]:
# Plot request_type in bar chart
result = df[df["request_type"] != ""]["request_type"].value_counts().reset_index()
result["count"] = result["count"] / result["count"].sum()
fig = px.bar(result, x="request_type", y="count", text="count")
# set width of plot
# add share as labels inside of bars, as percentage
fig.update_traces(
    textposition="inside", insidetextanchor="middle", texttemplate="%{text:.1%}"
)
fig.update_layout(width=1000)
# add y axis label
fig.update_yaxes(title_text="Share of tickets")
fig.show()

## Open Ticket analysis

In [ ]:
dfopen = df[df["status_category"] != "Fertig"].copy()
todays_date = pd.Timestamp.now(tz="UTC")
dfopen["days_open"] = (todays_date - dfopen["created"]).dt.days
dfopen["weeks_open"] = -np.floor(dfopen["days_open"] / 7)
result = (
    dfopen[df["status_category"] == "In Arbeit"][
        ["weeks_open", "Hauptkategorie", "status_category"]
    ]
    .groupby(["weeks_open", "Hauptkategorie"])
    .count()
    .reset_index()
)

# plot: weeks open on x axis, count of tickets per category on y axis as stacked vertical bars
fig = px.bar(result, x="weeks_open", y="status_category", color="Hauptkategorie")
fig.update_layout(width=1000)
fig.show()

In [ ]:
dfopen = df[df["status_category"] != "Fertig"].copy()
todays_date = pd.Timestamp.now(tz="UTC")
dfopen["days_open"] = (todays_date - dfopen["created"]).dt.days
dfopen["weeks_open"] = -np.floor(dfopen["days_open"] / 7)
result = (
    dfopen[df["status_category"] == "In Arbeit"][["weeks_open", "status", "key"]]
    .groupby(["weeks_open", "status"])
    .count()
    .reset_index()
)

# plot: weeks open on x axis, count of tickets per category on y axis as stacked vertical bars
fig = px.bar(result, x="weeks_open", y="key", color="status")
fig.update_layout(width=1000)
fig.show()

In [ ]:
result = (
    df[df["status_category"] != "Fertig"][["status_category", "status", "key"]]
    .groupby(["status_category", "status"])
    .count()
    .reset_index()
    .sort_values("key", ascending=False)
)
# plot using plotly with status_category on x axis and status on y axis
fig = px.bar(result, x="status_category", y="key", color="status", text="status")
# add labels inside of bars
fig.update_traces(textposition="inside", insidetextanchor="middle")
# set width of plot
fig.update_layout(width=1000)
fig.update_layout(
    title="Status and Status Category for open tickets",
)
# make bars in shades of grey
fig.update_traces(marker_color=px.colors.qualitative.Plotly)
# add y axis label
fig.update_yaxes(title_text="Count of tickets")
# remove legend
fig.update_layout(showlegend=False, uniformtext_minsize=10)
fig.show()

In [ ]:
df[df["status_category"] == "In Arbeit"]["status"].value_counts()

In [ ]:
df[df["priority"].isin(["Hoch", "Sehr Hoch", "Rot"])][
    ["Hauptkategorie", "priority"]
].groupby("Hauptkategorie").count().reset_index()

## Time to resolution analysis

In [ ]:
# plot time to resolution bin counts using plotly
# sort by midpoint of intervals/bins
result = (
    df[df["currentstatus_name"] == "Fertig"][["time_to_resolution_bin", "key"]]
    .groupby("time_to_resolution_bin")
    .count()
    .reset_index()
)
result["key"] = result["key"] / result["key"].sum()
fig = px.bar(result, x="time_to_resolution_bin", y="key")
# add x axis label

fig.update_layout(
    title="Share of tickets by time to resolution",
)
fig.update_xaxes(title_text="Time to resolution in hours")
# add y axis label
fig.update_yaxes(title_text="Share of tickets")
# set width of plot
fig.update_layout(width=1000)
fig.show()

In [ ]:
# plot time to resolution bin counts using plotly
# sort by midpoint of intervals/bins
result = (
    df[df["currentstatus_name"] == "Fertig"][["bdays"]].value_counts().reset_index()
)
result["count"] = result["count"] / result["count"].sum()
fig = px.bar(result, x="bdays", y="count")
# add x axis label)
fig.update_layout(
    title="Share of tickets by time to resolution in business days",
)
fig.update_xaxes(title_text="Time to resolution in business days")
# add y axis label
fig.update_yaxes(title_text="Share of tickets")
# set x axis range
fig.update_xaxes(range=[-1, 16])
fig.update_yaxes(range=[0, 0.7])
# set x tick values
fig.update_xaxes(tickvals=list(range(0, 16)))
# set width of plot
fig.update_layout(width=1000)
fig.show()

In [ ]:
# Analyze average time to resolution for top 15 main categories
# compute average time and count by main category
top15 = (
    df[df["currentstatus_name"] == "Fertig"]["Hauptkategorie"]
    .value_counts()
    .head(14)
    .index
)
df_top15 = df[df["Hauptkategorie"].isin(top15)]

result = df[
    (df["currentstatus_name"] == "Fertig") & (df["Hauptkategorie"].isin(top15))
][["Hauptkategorie", "time_to_resolution_h"]]

# group by main category and compute median time to resolution as well as count
result = (
    result.groupby("Hauptkategorie")
    .agg(
        count=("Hauptkategorie", "size"),
        median_time_delta=("time_to_resolution_h", "median"),
    )
    .reset_index()
    .sort_values("count", ascending=False)
)

# Plot the main categories with a percentage bar in terms of count and include the cumulative count as a line
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Bar trace
fig.add_trace(
    go.Bar(
        x=result["Hauptkategorie"],
        y=result["median_time_delta"],
        name="Median time to resolution in hours",
    ),
    secondary_y=False,
)

# Line trace (secondary y-axis)
fig.add_trace(
    go.Scatter(
        x=result["Hauptkategorie"],
        y=result["count"],
        mode="lines+markers",
        name="Count of tickets",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Median time to resolution in hours by main category (and count)",
    xaxis_title="Hauptkategorie",
    yaxis_title="Share of tickets",
)
fig.update_yaxes(title_text="Cumulative share", secondary_y=True)
# add x axis label in font size 8
fig.update_xaxes(title_text="Hauptkategorie", tickfont={"size": 8})
fig.update_xaxes(tickangle=45)
# add y axis label
fig.update_yaxes(title_text="Median time to resolution in hours")
# set width of plot
fig.update_layout(width=1000)
fig.show()

In [ ]:
result = (
    df[df["Hauptkategorie"].isin(top15)][["Hauptkategorie", "resolution", "key"]]
    .groupby(["Hauptkategorie", "resolution"])
    .count()
    .reset_index()
)
# sort by overall count
result = result.sort_values("key", ascending=False)
# normalize per Hauptkategorie
result = result.groupby("Hauptkategorie").apply(
    lambda x: x.assign(Share=x["key"] / x["key"].sum())
)

fig = px.bar(result, x="Hauptkategorie", y="Share", color="resolution")
fig.update_layout(width=1000)
fig.show()

In [ ]:
result = (
    df[["Hauptkategorie", "resolution", "key"]]
    .groupby(["Hauptkategorie", "resolution"])
    .count()
    .reset_index()
)
# sort by overall count
result = result.sort_values("key", ascending=False)

fig = px.bar(result, x="Hauptkategorie", y="key", color="resolution")
fig.update_layout(width=1000)
fig.show()

In [ ]:
result

In [ ]:
result = (
    df[df["currentstatus_name"] == "Fertig"][["Hauptkategorie", "key"]]
    .groupby("Hauptkategorie")
    .count()
    .reset_index()
    .sort_values("key", ascending=False)
)
result["Cumulative share"] = result["key"].cumsum() / result["key"].sum()
result["Share"] = result["key"] / result["key"].sum()


# Plot the main categories with a percentage bar in terms of count and include the cumulative count as a line
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Bar trace
fig.add_trace(
    go.Bar(
        x=result["Hauptkategorie"],
        y=result["Share"],
        name="Share",
    ),
    secondary_y=False,
)

# Line trace (secondary y-axis)
fig.add_trace(
    go.Scatter(
        x=result["Hauptkategorie"],
        y=result["Cumulative share"],
        mode="lines+markers",
        name="Cumulative share",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Distribution of tickets by main category",
    xaxis_title="Hauptkategorie",
    yaxis_title="Share of tickets",
)
fig.update_yaxes(title_text="Cumulative share", secondary_y=True)
# add x axis label in font size 8
fig.update_xaxes(title_text="Hauptkategorie", tickfont=dict(size=9))
fig.update_xaxes(tickangle=45)
# add y axis label
fig.update_yaxes(title_text="Share of tickets")
# set width of plot
fig.update_layout(width=1200)
fig.update_layout(height=500)
fig.show()

## Analyzing topics

In [ ]:
print("Top 15 main categories:", list(top15))

### Get tickets for September-November per category

In [ ]:
dff_oct2025 = df[(df["month"].isin([9, 10, 11])) & (df["Hauptkategorie"].isin(top15))]

In [ ]:
dff_oct2025.shape

In [ ]:
# Disabled: get_summary() lives in utils.py, which is not part of this
# repository. Restore utils.py to regenerate these LLM summaries.
# summaries = []
# for cat in top15:
#     descriptions = dff_oct2025[dff_oct2025["Hauptkategorie"] == cat]["description"]
#     titles = dff_oct2025[dff_oct2025["Hauptkategorie"] == cat]["summary"]
#     descriptions = [f"{t}\n{d}" for t, d in zip(titles, descriptions)]
#     summary = get_summary(descriptions)
#     summaries.append(f"## {cat}\n\n{summary}")
#
# summaries_markdown = "\n\n".join(summaries)
# # save summaries to file
# with open("summaries.md", "w") as f:
#     f.write(summaries_markdown)

In [ ]:
# Disabled: get_summary() lives in utils.py, which is not part of this
# repository. Restore utils.py to regenerate these LLM summaries.
# dff_oct2025_sameday = dff_oct2025[dff_oct2025["bdays"] == 0]
# summaries = []
# for cat in top15:
#     descriptions = dff_oct2025_sameday[dff_oct2025_sameday["Hauptkategorie"] == cat][
#         "description"
#     ]
#     titles = dff_oct2025_sameday[dff_oct2025_sameday["Hauptkategorie"] == cat][
#         "summary"
#     ]
#     descriptions = [f"{t}\n{d}" for t, d in zip(titles, descriptions)]
#     summary = get_summary(descriptions)
#     summaries.append(f"## {cat}\n\n{summary}")
#
# summaries_markdown = "\n\n".join(summaries)
# # save summaries to file
# with open("summaries_sameday.md", "w") as f:
#     f.write(summaries_markdown)

In [ ]:
# Disabled: get_summary() lives in utils.py, which is not part of this
# repository. Restore utils.py to regenerate these LLM summaries.
# (kept live - cell below still uses this frame)
dff_oct2025_emails = dff_oct2025[dff_oct2025["request_type"] == "Anfrage per E-Mail"]
# summaries = []
# for cat in top15:
#     descriptions = dff_oct2025_emails[dff_oct2025_emails["Hauptkategorie"] == cat][
#         "description"
#     ]
#     titles = dff_oct2025_emails[dff_oct2025_emails["Hauptkategorie"] == cat]["summary"]
#     descriptions = [f"{t}\n{d}" for t, d in zip(titles, descriptions)]
#     summary = get_summary(descriptions)
#     summaries.append(f"## {cat}\n\n{summary}")
#
# summaries_markdown = "\n\n".join(summaries)
# # save summaries to file
# with open("summaries_emails.md", "w") as f:
#     f.write(summaries_markdown)

In [ ]:
dff_oct2025_emails

## Focus on E-Mail tickets

In [ ]:
dfem = df[df["request_type"] == "Anfrage per E-Mail"]

In [ ]:
dfem.shape

## Clone relationships

In [ ]:
def get_clone_relations(issue):
    cloned_from = None
    clones = []

    for link in issue.fields.issuelinks:
        t = link.type

        # this issue was cloned FROM another
        if hasattr(link, "outwardIssue") and t.name == "Cloners":
            cloned_from = link.outwardIssue.key

        # this issue was cloned BY another
        if hasattr(link, "inwardIssue") and t.name == "Cloners":
            clones.append(link.inwardIssue.key)

    return cloned_from, clones

In [ ]:
issues2 = issues[:50]
for issue in issues2:
    issue = jira.issue(issue.key)
    cloned_from, clones = get_clone_relations(issue)

    if cloned_from is not None and cloned_from not in clones:
        print(issue.key, issue.fields.summary, "Cloned from:", cloned_from)
    if len(clones) > 0:
        print(issue.key, issue.fields.summary, "Clones:", clones)